# Week 8 live demo: relational databases
Toy tumor-cohort DB (not Lab 4's clinic.db). Run top to bottom.

In [ ]:
import sqlite3, pandas as pd
con = sqlite3.connect('demo_cohort.db')
def q(sql): return pd.read_sql(sql, con)
q("SELECT name FROM sqlite_master WHERE type='table'")

## SELECT / WHERE

In [ ]:
q("SELECT sample_id, subject_id, collected FROM samples WHERE tissue='tumor' ORDER BY collected")

## INNER vs LEFT: how many subjects survive?

In [ ]:
inner = q('SELECT s.subject_id, COUNT(sa.sample_id) n FROM subjects s JOIN samples sa ON sa.subject_id=s.subject_id GROUP BY s.subject_id')
left  = q('SELECT s.subject_id, COUNT(sa.sample_id) n FROM subjects s LEFT JOIN samples sa ON sa.subject_id=s.subject_id GROUP BY s.subject_id')
print(len(inner), 'vs', len(left)); left.tail(2)

## Grain: rows vs samples after a one-to-many join

In [ ]:
q('''SELECT s.subject_id, COUNT(*) n_rows, COUNT(DISTINCT sa.sample_id) n_samples
FROM subjects s JOIN samples sa ON sa.subject_id=s.subject_id
JOIN mutations m ON m.sample_id=sa.sample_id GROUP BY s.subject_id''')

## Mismatched keys: count before and after

In [ ]:
q('''SELECT (SELECT COUNT(*) FROM mutations) AS total,
(SELECT COUNT(*) FROM mutations m JOIN genes g ON g.symbol=m.gene_symbol) AS matched''')

In [ ]:
q('''SELECT m.mutation_id, '['||m.gene_symbol||']' AS symbol FROM mutations m
LEFT JOIN genes g ON g.symbol=m.gene_symbol WHERE g.gene_id IS NULL''')

In [ ]:
q('''SELECT COUNT(*) AS matched_after_fix FROM mutations m JOIN genes g ON g.symbol=UPPER(TRIM(m.gene_symbol))''')

## AI-style query: plausible but wrong

In [ ]:
bad = q('''SELECT s.subject_id, COUNT(DISTINCT sa.sample_id) AS kras_tumor_samples
FROM subjects s LEFT JOIN samples sa ON sa.subject_id=s.subject_id
LEFT JOIN mutations m ON m.sample_id=sa.sample_id
WHERE sa.tissue='tumor' AND m.gene_symbol='KRAS' GROUP BY s.subject_id''')
print(len(bad), 'rows'); bad

## Fixed: filters in ON, normalized symbol

In [ ]:
good = q('''SELECT s.subject_id, COUNT(DISTINCT m.sample_id) AS kras_tumor_samples
FROM subjects s
LEFT JOIN samples sa ON sa.subject_id=s.subject_id AND sa.tissue='tumor'
LEFT JOIN mutations m ON m.sample_id=sa.sample_id AND UPPER(TRIM(m.gene_symbol))='KRAS'
GROUP BY s.subject_id ORDER BY s.subject_id''')
print(len(good), 'rows'); good

## Genomic annotation join (real KRAS rows from UCSC hg38 MANE)

In [ ]:
q('''SELECT g.symbol, g.ncbi_gene_id, t.transcript_id, t.chrom, t.tx_start+1 AS start_1based, t.tx_end, t.strand, t.mane_status
FROM genes g JOIN transcripts t ON t.gene_id=g.gene_id WHERE g.symbol='KRAS' ''')